In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from scipy.signal import get_window


def init_kernels(win_len, win_inc, fft_len, win_type=None, invers=False):
    if win_type == 'None' or win_type is None:
        window = np.ones(win_len)
    else:
        window = get_window(win_type, win_len, fftbins=True)

    N = fft_len
    fourier_basis = np.fft.rfft(np.eye(N))[:win_len]
    real_kernel = np.real(fourier_basis)
    imag_kernel = np.imag(fourier_basis)
    kernel = np.concatenate([real_kernel, imag_kernel], 1).T

    if invers :
        kernel = np.linalg.pinv(kernel).T

    kernel = kernel*window
    kernel = kernel[:, None, :]
    return torch.from_numpy(kernel.astype(np.float32)), torch.from_numpy(window[None,:,None].astype(np.float32))


class ConvSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real', fix=True):
        super(ConvSTFT, self).__init__()

        if fft_len == None:
            self.fft_len = np.int(2**np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len

        kernel, _ = init_kernels(win_len, win_inc, self.fft_len, win_type)
        #self.weight = nn.Parameter(kernel, requires_grad=(not fix))
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.stride = win_inc
        self.win_len = win_len
        self.dim = self.fft_len

    def forward(self, inputs):
        if inputs.dim() == 2:
            inputs = torch.unsqueeze(inputs, 1)
        inputs = F.pad(inputs,[self.win_len-self.stride, self.win_len-self.stride])
        outputs = F.conv1d(inputs, self.weight, stride=self.stride)
        dim = self.dim//2+1
        real = outputs[:, :dim, :]
        imag = outputs[:, dim:, :]
        mags = torch.sqrt(real**2+imag**2)
        phase = torch.atan2(imag, real)
        return mags, phase, outputs

class ConviSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real', fix=True):
        super(ConviSTFT, self).__init__()
        if fft_len == None:
            self.fft_len = np.int(2**np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len
        kernel, window = init_kernels(win_len, win_inc, self.fft_len, win_type, invers=True)
        #self.weight = nn.Parameter(kernel, requires_grad=(not fix))
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.win_type = win_type
        self.win_len = win_len
        self.stride = win_inc
        self.stride = win_inc
        self.dim = self.fft_len
        self.register_buffer('window', window)
        self.register_buffer('enframe', torch.eye(win_len)[:,None,:])

    def forward(self, inputs, phase = None):
        """
        inputs : [B, N+2, T] (complex spec)
        or [B, N//2+1, T] (mags)
        phase: [B, N//2+1, T] (if not none)
        """

        if phase is not None:
            real = inputs*torch.cos(phase)
            imag = inputs*torch.sin(phase)
            inputs = torch.cat([real, imag], 1)
        outputs = F.conv_transpose1d(inputs, self.weight, stride=self.stride)

        # this is from torch-stft: https://github.com/pseeth/torch-stft
        t = self.window.repeat(1,1,inputs.size(-1))**2
        coff = F.conv_transpose1d(t, self.enframe, stride=self.stride)
        outputs = outputs/(coff+1e-8)
        #outputs = torch.where(coff == 0, outputs, outputs/coff)
        outputs = outputs[...,self.win_len-self.stride:-(self.win_len-self.stride)]

        return outputs


import librosa
import soundfile as sf








In [2]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import os
import sys

def mse_loss():
    return torch.nn.MSELoss()

def l1_loss():
    return torch.nn.L1Loss()

def CE_loss():
    return torch.nn.CrossEntropyLoss()

class GLayerNorm2d(nn.Module):

    def __init__(self, in_channel, eps=1e-12):
        super(GLayerNorm2d, self).__init__()
        self.eps = eps
        self.beta = nn.Parameter(torch.ones([1, in_channel,1,1]))
        self.gamma = nn.Parameter(torch.zeros([1, in_channel,1,1]))

    def forward(self,inputs):
        mean = torch.mean(inputs,[1,2,3], keepdim=True)
        var = torch.var(inputs,[1,2,3], keepdim=True)
        outputs = (inputs - mean)/ torch.sqrt(var+self.eps)*self.beta+self.gamma
        return outputs

class Axial_Layer(nn.Module):
    def __init__(self, in_channels, num_heads=8, kernel_size=7, stride=1, height_dim=True, inference=False):
        super(Axial_Layer, self).__init__()
        self.depth = in_channels
        self.num_heads = num_heads
        self.kernel_size = kernel_size
        self.stride = stride
        self.height_dim = height_dim
        self.dh = self.depth // self.num_heads

        assert self.depth % self.num_heads == 0, "depth should be divided by num_heads. (example: depth: 32, num_heads: 8)"

        self.kqv_conv = nn.Conv1d(in_channels, self.depth * 2, kernel_size=1, bias=False)
        self.kqv_bn = nn.BatchNorm1d(self.depth * 2)
        self.logits_bn = nn.BatchNorm2d(num_heads * 3)
        # Positional encodings
        self.rel_encoding = nn.Parameter(torch.randn(self.dh * 2, kernel_size * 2 - 1), requires_grad=True)
        key_index = torch.arange(kernel_size)
        query_index = torch.arange(kernel_size)
        # Shift the distance_matrix so that it is >= 0. Each entry of the
        # distance_matrix distance will index a relative positional embedding.
        distance_matrix = (key_index[None, :] - query_index[:, None]) + kernel_size - 1
        self.register_buffer('distance_matrix', distance_matrix.reshape(kernel_size*kernel_size))

        # later access attention weights
        self.inference = inference
        if self.inference:
            self.register_parameter('weights', None)

    def forward(self, x):

        if self.height_dim:
            x = x.permute(0, 3, 1, 2)  # batch_size, width, depth, height
        else:
            x = x.permute(0, 2, 1, 3)  # batch_size, height, depth, width

        batch_size, width, depth, height = x.size()
        x = x.reshape(batch_size * width, depth, height)
        # Compute q, k, v
        kqv = self.kqv_conv(x)
        kqv = self.kqv_bn(kqv) # apply batch normalization on k, q, v
        k, q, v = torch.split(kqv.reshape(batch_size * width, self.num_heads, self.dh * 2, height), [self.dh // 2, self.dh // 2, self.dh], dim=2)
        # Positional encodings
        rel_encodings = torch.index_select(self.rel_encoding, 1, self.distance_matrix).reshape(self.dh * 2, self.kernel_size, self.kernel_size)
        q_encoding, k_encoding, v_encoding = torch.split(rel_encodings, [self.dh // 2, self.dh // 2, self.dh], dim=0)
        # qk + qr + kr
        qk = torch.matmul(q.transpose(2, 3), k)
        qr = torch.einsum('bhdx,dxy->bhxy', q, q_encoding)
        kr = torch.einsum('bhdx,dxy->bhxy', k, k_encoding).transpose(2, 3)
        logits = torch.cat([qk, qr, kr], dim=1)
        logits = self.logits_bn(logits) # apply batch normalization on qk, qr, kr
        logits = logits.reshape(batch_size * width, 3, self.num_heads, height, height).sum(dim=1)
        weights = F.softmax(logits, dim=3)

        if self.inference:
            self.weights = nn.Parameter(weights)

        attn = torch.matmul(weights, v.transpose(2,3)).transpose(2,3)
        attn_encoding = torch.einsum('bhxy,dxy->bhdx', weights, v_encoding)
        attn_out = torch.cat([attn, attn_encoding], dim=-1).reshape(batch_size * width, self.depth * 2, height)
        output = attn_out.reshape(batch_size, width, self.depth, 2, height).sum(dim=-2)

        if self.height_dim:
            output = output.permute(0, 2, 3, 1)
        else:
            output = output.permute(0, 2, 1, 3)

        return output

class Axial_Layer_cross(nn.Module):
    def __init__(self, in_channels, num_heads=8, kernel_size=7, stride=1, height_dim=True, inference=False):
        super(Axial_Layer_cross, self).__init__()
        self.depth = in_channels
        self.num_heads = num_heads
        self.kernel_size = kernel_size
        self.stride = stride
        self.height_dim = height_dim
        self.dh = self.depth // self.num_heads

        assert self.depth % self.num_heads == 0, "depth should be divided by num_heads. (example: depth: 32, num_heads: 8)"

        self.v_conv = nn.Conv1d(in_channels, self.depth, kernel_size=1, bias=False)
        self.v_bn = nn.BatchNorm1d(self.depth)

        self.q_conv = nn.Conv1d(in_channels, self.depth // 2, kernel_size=1, bias=False)
        self.q_bn = nn.BatchNorm1d(self.depth // 2)

        self.k_conv = nn.Conv1d(in_channels, self.depth // 2, kernel_size=1, bias=False)
        self.k_bn = nn.BatchNorm1d(self.depth // 2)


        self.kq_conv = nn.Conv1d(in_channels, self.depth, kernel_size=1, bias=False)
        self.kq_bn = nn.BatchNorm1d(self.depth)

        self.logits_bn = nn.BatchNorm2d(num_heads * 3)
        # Positional encodings
        self.rel_encoding = nn.Parameter(torch.randn(self.dh * 2, kernel_size * 2 - 1), requires_grad=True)
        key_index = torch.arange(kernel_size)
        query_index = torch.arange(kernel_size)
        # Shift the distance_matrix so that it is >= 0. Each entry of the
        # distance_matrix distance will index a relative positional embedding.
        distance_matrix = (key_index[None, :] - query_index[:, None]) + kernel_size - 1
        self.register_buffer('distance_matrix', distance_matrix.reshape(kernel_size*kernel_size))

        # later access attention weights
        self.inference = inference
        if self.inference:
            self.register_parameter('weights', None)

    def forward(self, x, y):
        if self.height_dim:
            x = x.permute(0, 3, 1, 2)  # batch_size, width, depth, height
            y = y.permute(0, 3, 1, 2)  # batch_size, width, depth, height
        else:
            x = x.permute(0, 2, 1, 3)  # batch_size, height, depth, width
            y = y.permute(0, 2, 1, 3)  # batch_size, height, depth, width

        batch_size, width, depth, height = x.size()
        x = x.reshape(batch_size * width, depth, height)
        y = y.reshape(batch_size * width, depth, height)
        # Compute q, k, v
        k = self.k_conv(x)
        k = self.k_bn(k) # apply batch normalization on k, q, v

        v = self.v_conv(x)
        v = self.kq_bn(v) # apply batch normalization on k, q, v

        q = self.q_conv(y)
        q = self.q_bn(q) # apply batch normalization on k, q, v

        kqv = torch.cat([k, q, v], dim = 1)
        k, q, v = torch.split(kqv.reshape(batch_size * width, self.num_heads, self.dh * 2, height), [self.dh // 2, self.dh // 2, self.dh], dim=2)
        #q = q.reshape(batch_size * width, self.num_heads, self.dh, height)

        # Positional encodings
        rel_encodings = torch.index_select(self.rel_encoding, 1, self.distance_matrix).reshape(self.dh * 2, self.kernel_size, self.kernel_size)
        q_encoding, k_encoding, v_encoding = torch.split(rel_encodings, [self.dh // 2, self.dh // 2, self.dh], dim=0)

        # qk + qr + kr
        qk = torch.matmul(q.transpose(2, 3), k)
        qr = torch.einsum('bhdx,dxy->bhxy', q, q_encoding)
        kr = torch.einsum('bhdx,dxy->bhxy', k, k_encoding).transpose(2, 3)

        logits = torch.cat([qk, qr, kr], dim=1)
        logits = self.logits_bn(logits) # apply batch normalization on qk, qr, kr
        logits = logits.reshape(batch_size * width, 3, self.num_heads, height, height).sum(dim=1)

        weights = F.softmax(logits, dim=3)

        if self.inference:
            self.weights = nn.Parameter(weights)

        attn = torch.matmul(weights, v.transpose(2,3)).transpose(2,3)
        attn_encoding = torch.einsum('bhxy,dxy->bhdx', weights, v_encoding)
        attn_out = torch.cat([attn, attn_encoding], dim=-1).reshape(batch_size * width, self.depth * 2, height)
        output = attn_out.reshape(batch_size, width, self.depth, 2, height).sum(dim=-2)

        if self.height_dim:
            output = output.permute(0, 2, 3, 1)
        else:
            output = output.permute(0, 2, 1, 3)

        return output

class CausalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 2),
            stride=(2, 1),
            padding=(0, 1)
        )
        self.norm = nn.BatchNorm2d(num_features=out_channels)
        self.activation = nn.ELU()

    def forward(self, x):
        """
        2D Causal convolution.
        Args:
            x: [B, C, F, T]

        Returns:
            [B, C, F, T]
        """
        x = self.conv(x)
        x = x[:, :, :, :-1]  # chomp size
        x = self.norm(x)
        x = self.activation(x)
        return x


class CausalTransConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, is_last=False, output_padding=(0, 0)):
        super().__init__()
        self.conv = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 2),
            stride=(2, 1),
            output_padding=output_padding
        )
        self.norm = nn.BatchNorm2d(num_features=out_channels)
        if is_last:
            self.activation = nn.ReLU()
        else:
            self.activation = nn.ELU()

    def forward(self, x):
        """
        2D Causal convolution.
        Args:
            x: [B, C, F, T]

        Returns:
            [B, C, F, T]
        """
        x = self.conv(x)
        x = x[:, :, :, :-1]  # chomp size
        x = self.norm(x)
        x = self.activation(x)
        return x

class Model(nn.Module):

    def __init__(self, channel_amp = 1, channel_phase=3):
        super(Model, self).__init__()
        self.stft = ConvSTFT(512, 256, 512, 'hamming', 'real', True)
        self.istft = ConviSTFT(512, 256, 512, 'hamming', 'real', True)

        # Encoder
        self.conv_block_1 = CausalConvBlock(1, 16)
        self.conv_block_2 = CausalConvBlock(16, 32)
        self.conv_block_3 = CausalConvBlock(32, 64)
        self.conv_block_4 = CausalConvBlock(64, 128)
        self.conv_block_5 = CausalConvBlock(128, 256)

        self.conv_block_1ref = CausalConvBlock(1, 16)
        self.conv_block_2ref = CausalConvBlock(16, 32)
        self.conv_block_3ref = CausalConvBlock(32, 64)
        self.conv_block_4ref = CausalConvBlock(64, 128)
        self.conv_block_5ref = CausalConvBlock(128, 256)

        self.SA_time = Axial_Layer(256, height_dim=True)
        self.SA_frequency = Axial_Layer(256, kernel_size=126, height_dim = False)


        self.CA_time_1 = Axial_Layer_cross(256, height_dim=True)
        self.CA_time_2 = Axial_Layer_cross(128, kernel_size=15, height_dim=True)
        self.CA_time_3 = Axial_Layer_cross(64, kernel_size=31, height_dim=True)
        self.CA_time_4 = Axial_Layer_cross(32, kernel_size=63, height_dim=True)
        self.CA_time_5 = Axial_Layer_cross(16, kernel_size=128, height_dim=True)

        self.CA_time_11 = Axial_Layer_cross(256, height_dim=True)
        self.CA_time_22 = Axial_Layer_cross(128, kernel_size=15, height_dim=True)
        self.CA_time_33 = Axial_Layer_cross(64, kernel_size=31, height_dim=True)
        self.CA_time_44 = Axial_Layer_cross(32, kernel_size=63, height_dim=True)
        self.CA_time_55 = Axial_Layer_cross(16, kernel_size=128, height_dim=True)

        self.CA_frequency_1 = Axial_Layer_cross(256, kernel_size=126, height_dim = False)
        self.CA_frequency_2 = Axial_Layer_cross(128, kernel_size=126, height_dim = False)
        self.CA_frequency_3 = Axial_Layer_cross(64, kernel_size=126, height_dim = False)
        self.CA_frequency_4 = Axial_Layer_cross(32, kernel_size=126, height_dim = False)
        self.CA_frequency_5 = Axial_Layer_cross(16, kernel_size=126, height_dim = False)

        self.CA_frequency_11 = Axial_Layer_cross(256, kernel_size=126, height_dim = False)
        self.CA_frequency_22 = Axial_Layer_cross(128, kernel_size=126, height_dim = False)
        self.CA_frequency_33 = Axial_Layer_cross(64, kernel_size=126, height_dim = False)
        self.CA_frequency_44 = Axial_Layer_cross(32, kernel_size=126, height_dim = False)
        self.CA_frequency_55 = Axial_Layer_cross(16, kernel_size=126, height_dim = False)

        self.tran_conv_block_1 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_2 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_3 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_4 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_5 = CausalTransConvBlock(16 + 16, 1, is_last=True)


        self.tran_conv_block_11 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_22 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_33 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_44 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_55 = CausalTransConvBlock(16 + 16, 1, is_last=True)

        self.tran_conv_block_111 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_222 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_333 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_444 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_555 = CausalTransConvBlock(16 + 16, 1, is_last=True)

        self.tran_conv_block_1111 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_2222 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_3333 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_4444 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_5555 = CausalTransConvBlock(16 + 16, 1, is_last=True)


    def forward(self, x, ref):
        amp_spec, phase_spec, comp_spec = self.stft(x)
        ref_amp_spec, ref_phase_spec, ref_comp_spec = self.stft(ref)

        phase_spec = torch.unsqueeze(phase_spec, 1)
        amp_spec = torch.unsqueeze(amp_spec, 1)
        ref_phase_spec = torch.unsqueeze(ref_phase_spec, 1)
        ref_amp_spec = torch.unsqueeze(ref_amp_spec, 1)
        # amp_spec,phase_spec  = torch.Size([1, 1, 257, 251])


        e1 = self.conv_block_1(amp_spec)
        e2 = self.conv_block_2(e1)
        e3 = self.conv_block_3(e2)
        e4 = self.conv_block_4(e3)
        e5 = self.conv_block_5(e4)



        refe1 = self.conv_block_1ref(ref_amp_spec)
        refe2 = self.conv_block_2ref(refe1)
        refe3 = self.conv_block_3ref(refe2)
        refe4 = self.conv_block_4ref(refe3)
        refe5 = self.conv_block_5ref(refe4)


        b_time = self.SA_time(e5)
        b_frequency = self.SA_frequency(e5)
        b = e5 + b_time + b_frequency


        e5_1 = self.CA_time_1(e5, b)
        refe5_1 = self.CA_time_11(refe5, b)

        e5_2 = self.CA_frequency_1(e5, b)
        refe5_2= self.CA_frequency_11(refe5, b)

        d1 = self.tran_conv_block_1(torch.cat([b, e5_1], 1))
        d2 = self.tran_conv_block_11(torch.cat([b, refe5_1], 1))
        d3 = self.tran_conv_block_111(torch.cat([b, e5_2], 1))
        d4 = self.tran_conv_block_1111(torch.cat([b, refe5_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e4_1 = self.CA_time_2(e4, d)
        refe4_1 = self.CA_time_22(refe4, d)

        e4_2 = self.CA_frequency_2(e4, d)
        refe4_2= self.CA_frequency_22(refe4, d)

        d1 = self.tran_conv_block_2(torch.cat([d, e4_1], 1))
        d2 = self.tran_conv_block_22(torch.cat([d, refe4_1], 1))
        d3 = self.tran_conv_block_222(torch.cat([d, e4_2], 1))
        d4 = self.tran_conv_block_2222(torch.cat([d, refe4_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e3_1 = self.CA_time_3(e3, d)
        refe3_1 = self.CA_time_33(refe3, d)

        e3_2 = self.CA_frequency_3(e3, d)
        refe3_2= self.CA_frequency_33(refe3, d)

        d1 = self.tran_conv_block_3(torch.cat([d, e3_1], 1))
        d2 = self.tran_conv_block_33(torch.cat([d, refe3_1], 1))
        d3 = self.tran_conv_block_333(torch.cat([d, e3_2], 1))
        d4 = self.tran_conv_block_3333(torch.cat([d, refe3_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e2_1 = self.CA_time_4(e2, d)
        refe2_1 = self.CA_time_44(refe2, d)

        e2_2 = self.CA_frequency_4(e2, d)
        refe2_2= self.CA_frequency_44(refe2, d)

        d1 = self.tran_conv_block_4(torch.cat([d, e2_1], 1))
        d2 = self.tran_conv_block_44(torch.cat([d, refe2_1], 1))
        d3 = self.tran_conv_block_444(torch.cat([d, e2_2], 1))
        d4 = self.tran_conv_block_4444(torch.cat([d, refe2_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e1_1 = self.CA_time_5(e1, d)
        refe1_1 = self.CA_time_55(refe1, d)

        e1_2 = self.CA_frequency_5(e1, d)
        refe1_2= self.CA_frequency_55(refe1, d)

        d1 = self.tran_conv_block_5(torch.cat([d, e1_1], 1))
        d2 = self.tran_conv_block_55(torch.cat([d, refe1_1], 1))
        d3 = self.tran_conv_block_555(torch.cat([d, e1_2], 1))
        d4 = self.tran_conv_block_5555(torch.cat([d, refe1_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        p5 = phase_spec

        p5 = torch.squeeze(p5, 1)
        d = torch.squeeze(d, 1)

        real = d*torch.cos(p5)
        imag = d*torch.sin(p5)
        est_spec = torch.cat([real, imag], 1)
        est_wav = self.istft(est_spec, None)
        est_wav = torch.squeeze(est_wav, 1)
        return est_spec, est_wav



    def loss_snr(self, est, labels):
        if labels.dim() == 3:
            labels = torch.squeeze(labels,1)
        if est.dim() == 3:
            est = torch.squeeze(est,1)
        return -si_snr(est, labels)



def remove_dc(data):
    mean = torch.mean(data, -1, keepdim=True)
    data = data - mean
    return data
def l2_norm(s1, s2):
    #norm = torch.sqrt(torch.sum(s1*s2, 1, keepdim=True))
    #norm = torch.norm(s1*s2, 1, keepdim=True)

    norm = torch.sum(s1*s2, -1, keepdim=True)
    return norm

def si_snr(s1, s2, eps=1e-8):

    # Ensure inputs are PyTorch tensors
    if not isinstance(s1, torch.Tensor):
        s1 = torch.tensor(s1)
    if not isinstance(s2, torch.Tensor):
        s2 = torch.tensor(s2)

    if s1.dim() == 3:
        s1 = torch.squeeze(s1,1)
    if s2.dim() == 3:
        s2 = torch.squeeze(s2,1)

    s1_s2_norm = l2_norm(s1, s2)
    s2_s2_norm = l2_norm(s2, s2)
    s_target = s1_s2_norm / (s2_s2_norm + eps) * s2
    e_nosie = s1 - s_target  # Now both are tensors
    target_norm = l2_norm(s_target, s_target)
    noise_norm = l2_norm(e_nosie, e_nosie)
    snr = 10 * torch.log10((target_norm) / (noise_norm + eps) + eps)
    return torch.mean(snr)


def test_selfattention1():
    torch.manual_seed(20)

    inputs = torch.randn([1, 2 * 16000])
    ref = torch.randn([1, 2 * 16000])
    net = Model()

    out1, out2 = net(inputs, ref)
    print(out1.shape)
    print(out2.shape)



In [3]:
import os
import torch
import librosa
import soundfile as sf
from torch.utils import data
import random
from pathlib import Path


def sample_fixed_length_data_aligned(data_a, data_b, sample_length):
    """sample with fixed length from two dataset
    """
    # Replicate data_a and data_b until their lengths are >= sample_length
    while len(data_a) < sample_length:
        data_a = np.concatenate((data_a, data_a), axis=0)
        data_b = np.concatenate((data_b, data_b), axis=0)

    frames_total = len(data_a)

    start = np.random.randint(frames_total - sample_length + 1)
    # print(f"Random crop from: {start}")
    end = start + sample_length

    return data_a[start:end], data_b[start:end]


class Dataset(data.Dataset):
    def __init__(self,
                 dataset,
                 limit,
                 PreModel,
                 offset=0,
                 sample_length=32000,
                 mode="train",
                 enhanced_dir="./enhanced"):
        """
        Construct dataset for training and validation.
        """
        super(Dataset, self).__init__()
        dataset_list = [line.rstrip('\n') for line in open(os.path.abspath(os.path.expanduser(dataset)), "r")]
        self.dataset_list = dataset_list


        assert mode in ("train", "validation"), "Mode must be one of 'train' or 'validation'."

        self.length = len(self.dataset_list)
        self.sample_length = sample_length
        self.mode = mode
        self.device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

        self.PreModel = PreModel.to(self.device)
        self.reload_premodel()
        self.PreModel.eval()  # inference mode

        os.makedirs(enhanced_dir, exist_ok=True)


        # Build new dataset list with enhanced paths
        new_dataset_list = []
        i = 0
        for line in self.dataset_list:
            i += 1
            if i % 500 == 0:
                print(f"Processed {i} lines")

            noisy_path, clean_path = line.split(" ")
            filename = os.path.splitext(os.path.basename(noisy_path))[0]
            enhanced_path = os.path.join(enhanced_dir, f"{filename}_enhanced.wav")

            # Only compute if not already saved
            if not os.path.exists(enhanced_path):
                noisy, _ = librosa.load(os.path.abspath(os.path.expanduser(noisy_path)), sr=16000)
                original_len = len(noisy)

                # Pad to multiple of sample_length
                pad_len = self.sample_length - (len(noisy) % self.sample_length)
                noisy_padded = np.pad(noisy, (0, pad_len), mode='constant')

                # Split into chunks
                chunks = noisy_padded.reshape(-1, self.sample_length)

                person_id = clean_path.split('/')[-1].split('_')[0]
                same_person_clean_paths = [line.split()[1] for line in self.dataset_list if person_id in line]
                same_person_clean_paths.remove(clean_path)
                clean_ref_path = random.choice(same_person_clean_paths)
                aligned_clean_ref, _ = librosa.load(os.path.abspath(os.path.expanduser(clean_ref_path)), sr=16000)

                enhanced_chunks = []
                with torch.inference_mode():
                    for chunk in chunks:


                        mixture2, clean_ref1 = sample_fixed_length_data_aligned(aligned_clean_ref, aligned_clean_ref, self.sample_length)

                        chunk_tensor = torch.tensor(chunk, dtype=torch.float32, device=self.device).unsqueeze(0).unsqueeze(0)
                        ref1 = torch.tensor(clean_ref1, dtype=torch.float32, device=self.device).unsqueeze(0).unsqueeze(0)
                        sp, enhanced_tensor = self.PreModel(chunk_tensor, ref1) # Pass ref1 as the second argument
                        enhanced_chunk = enhanced_tensor.squeeze().cpu().numpy()
                        enhanced_chunks.append(enhanced_chunk)

                enhanced_full = np.concatenate(enhanced_chunks)[:original_len]
                sf.write(enhanced_path, enhanced_full, 16000)


            new_dataset_list.append(f"{noisy_path} {clean_path} {enhanced_path}")

        self.dataset_list = new_dataset_list


    def reload_premodel(self):
        PreModel_root = Path("/home/jovyan/Experiments/WaveUNet").expanduser().absolute() / 'config'
        PreModel_checkpoints_dir = PreModel_root / "checkpoints"

        latest_model_path = PreModel_checkpoints_dir.expanduser().absolute() / "latest_model.tar"
        assert latest_model_path.exists(), f"{latest_model_path} does not exist, can not load PreModel checkpoint."

        checkpoint = torch.load(latest_model_path.as_posix(), map_location=self.device)

        # self.start_epoch = checkpoint["epoch"] + 1
        # self.best_score = checkpoint["best_score"]
        # self.optimizer.load_state_dict(checkpoint["optimizer"])

        if isinstance(self.PreModel, torch.nn.DataParallel):
            self.PreModel.module.load_state_dict(checkpoint["model"])
        else:
            self.PreModel.load_state_dict(checkpoint["model"])

    def __len__(self):
        return self.length

    def __getitem__(self, item):
        parts = self.dataset_list[item].split(" ")
        mixture_path, clean_path, enhanced_path = parts
        filename = os.path.splitext(os.path.basename(mixture_path))[0]

        mixture, _ = librosa.load(os.path.abspath(os.path.expanduser(mixture_path)), sr=16000)
        clean, _ = librosa.load(os.path.abspath(os.path.expanduser(clean_path)), sr=16000)
        enhanced, _ = librosa.load(os.path.abspath(os.path.expanduser(enhanced_path)), sr=16000)

        return mixture, clean, enhanced, filename

In [4]:
Pre_model = Model()


train_dataset = Dataset(
        dataset="/home/jovyan/file_paths_combined.txt",
        limit=3,
        PreModel=Pre_model,
        offset=0,
        sample_length=32000,
        mode="train"
    )

Processed 500 lines
Processed 1000 lines
Processed 1500 lines
Processed 2000 lines
Processed 2500 lines
Processed 3000 lines
Processed 3500 lines
Processed 4000 lines
Processed 4500 lines
Processed 5000 lines
Processed 5500 lines
Processed 6000 lines
Processed 6500 lines
Processed 7000 lines
Processed 7500 lines
Processed 8000 lines
Processed 8500 lines
Processed 9000 lines
Processed 9500 lines
Processed 10000 lines
Processed 10500 lines
Processed 11000 lines
Processed 11500 lines
Processed 12000 lines


In [ ]:
import shutil

src = "/home/jovyan/30epochPremodel/WaveUNet/config/checkpoints/latest_model.tar"
dst = "/home/jovyan/latest_model.tar"


try:
    shutil.copyfile(src, dst)
    print("File copied successfully!")
except FileExistsError:
    print("The destination folder already exists. Please specify a new destination folder.")
except Exception as e:
    print(f"An error occurred: {e}")
    

File copied successfully!
